# 01. OpenRouter LLM Client Complete Lab (`src/core/llm.py` 전체 구현)

이 노트북은 `src/core/llm.py`에 포함된 **모든 상수, 타입, 헬퍼, 퍼블릭 함수(`get_client`, `_build_messages`, `chat`, `chat_structured`, `chat_vision`)**를 빠짐없이 직접 타이핑하며 구현·검증하는 완전판 학습 레퍼런스야.

### 사전 준비
1. 터미널에서 `uv add --dev ipykernel` 실행
2. 우측 상단 커널 선택에서 `.venv` (`Python 3.13`) 선택
3. `Step 0`부터 `Step 6`까지 마크다운의 레퍼런스 코드와 해설을 보며 아래 빈 코드 셀에 직접 입력·실행해 봐.

## Step 0. 환경 확인, 임포트 및 전역 상수/타입 정의

### 목적 및 의의
* **노트북 실행 환경 보정**: `sys.path`에 프로젝트 루트를 등록해 이후 `src` 패키지 임포트가 가능하게 하고, `OPENROUTER_API_KEY` 로드 상태를 안전하게 확인한다.
* **모듈 전역 계약(Contract) 선언**: `llm.py` 전체에서 공유할 엔드포인트 URL, 기본 모델명, 제네릭 타입 변수 `T`를 상단에 정의한다.

### 레퍼런스 코드
```python
from __future__ import annotations

import base64
import mimetypes
import os
import sys
from pathlib import Path
from typing import TypeVar

from openai import OpenAI
from pydantic import BaseModel

# 노트북 경로 보정 (실제 llm.py 파일에는 불필요하지만 주피터 실습용으로 추가)
cwd = Path.cwd()
project_root = cwd.parent if (cwd.parent / "src").is_dir() else cwd
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
DEFAULT_MODEL = os.environ.get("OPENROUTER_MODEL", "google/gemini-2.5-flash")
T = TypeVar("T", bound=BaseModel)

print("project_root:", project_root)
print("DEFAULT_MODEL:", DEFAULT_MODEL)
print("API key loaded:", bool(os.environ.get("OPENROUTER_API_KEY")))
```

### 라인별 상세 해설
* `from __future__ import annotations`: PEP 563 지연 평가 타입 어노테이션 활성화 (`str | None` 등 유니온 문법 및 전방 참조를 런타임 평가 오버헤드 없이 사용).
* `import base64`: `chat_vision`에서 로컬 이미지 바이너리를 RFC 2397 Data URI용 Base64 ASCII 문자열로 인코딩하기 위해 임포트.
* `import mimetypes`: `chat_vision`에서 이미지 파일 확장자(`.png`, `.jpg` 등)를 보고 MIME 타입을 자동 판별하기 위해 임포트.
* `import os`: 셸 환경변수(`OPENROUTER_API_KEY`, `OPENROUTER_MODEL`)를 런타임에 조회하기 위해 임포트.
* `from pathlib import Path`: OS 독립적인 객체 지향 파일 시스템 경로 조작(`.is_file()`, `.read_bytes()`)을 위해 임포트.
* `from typing import TypeVar`: `chat_structured`에서 호출자가 전달한 Pydantic 모델 타입을 반환 타입으로 그대로 보존(제네릭)하기 위해 임포트.
* `from openai import OpenAI`: OpenRouter가 OpenAI Chat Completions API 스펙과 호환되므로 공식 SDK를 그대로 활용.
* `from pydantic import BaseModel`: Structured Outputs의 스키마 기반 클래스.
* `OPENROUTER_BASE_URL`: OpenAI 기본 주소 대신 사용할 OpenRouter 게이트웨이 루트 URL 상수.
* `DEFAULT_MODEL`: 환경변수 `OPENROUTER_MODEL`이 설정돼 있으면 우선 사용하고, 없으면 `google/gemini-2.5-flash`를 기본값으로 채택.
* `T = TypeVar("T", bound=BaseModel)`: `BaseModel`의 서브클래스만 타입 인자로 바인딩할 수 있도록 상한(upper bound)을 건 제네릭 타입 변수.

In [11]:
from __future__ import annotations
from dotenv import load_dotenv
load_dotenv()
import mimetypes
import os, sys, base64; print(sys.executable)
from pathlib import Path
from typing import TypeVar
from openai import OpenAI
from pydantic import BaseModel
cwd = Path.cwd(); print(cwd)
project_root = cwd.parent if (cwd.parent / 'src').is_dir() else cwd; print(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
DEFAULT_MODEL = os.environ.get("OPENROUTER_MODEL", "google/gemini-2.5-flash")
T=TypeVar("T", bound=BaseModel)

print("project_root:", project_root)
print("DEFAULT_MODEL:", DEFAULT_MODEL)
print("API key loaded:", bool(os.environ.get("OPENROUTER_API_KEY")))

/Users/jmahn/project/code/rag-cag/.venv/bin/python3
/Users/jmahn/project/code/rag-cag/notebooks
/Users/jmahn/project/code/rag-cag
project_root: /Users/jmahn/project/code/rag-cag
DEFAULT_MODEL: google/gemini-2.5-flash
API key loaded: True


## Step 1. `get_client()` — OpenRouter 클라이언트 팩토리

### 목적 및 의의
* **클라이언트 초기화 단일화 & Fail-Fast**: API 키 탐색 우선순위(인자 > 환경변수)를 통일하고, 키가 없을 때 네트워크 요청 전에 명확한 `RuntimeError`로 조기 실패시킨다.

### 레퍼런스 코드
```python
def get_client(api_key: str | None = None) -> OpenAI:
    resolved_key = api_key or os.environ.get("OPENROUTER_API_KEY")
    if not resolved_key:
        raise RuntimeError(
            "OPENROUTER_API_KEY is not set. Export it in your shell or .env file."
        )

    return OpenAI(
        base_url=OPENROUTER_BASE_URL,
        api_key=resolved_key,
        default_headers={
            "X-Title": "rag-cag",
        },
    )


client = get_client()
print("client.base_url:", client.base_url)
```

### 라인별 상세 해설
* `def get_client(api_key: str | None = None) -> OpenAI:`: 테스트 시 더미 키를 직접 주입하거나(`get_client("sk-...")`), 운영 시 환경변수를 자동 참조하도록 선택적 `api_key` 파라미터 설계.
* `resolved_key = api_key or os.environ.get("OPENROUTER_API_KEY")`: 단락 평가(`or`)를 이용해 명시적 인자가 우선이고, 없으면 환경변수를 폴백으로 조회.
* `if not resolved_key: raise RuntimeError(...)`: 빈 문자열이나 `None`인 채로 API를 호출해 401 HTTP 에러를 뒤늦게 만나는 대신, 설정 누락 원인을 즉시 알려주는 방어 로직.
* `base_url=OPENROUTER_BASE_URL`: SDK의 목적지를 OpenAI에서 OpenRouter로 치환.
* `default_headers={"X-Title": "rag-cag"}`: OpenRouter 대시보드/사용량 로그에서 프로젝트 출처를 식별하기 위한 HTTP 헤더.

In [12]:
def get_client(api_key: str | None = None) ->OpenAI:
    resolved_key= api_key or os.environ.get("OPENROUTER_API_KEY")
    if not resolved_key:
        raise RuntimeError(
            "OPENROUTER_API_KEY is not set. Export it in your shell or .env file."
        )
    return OpenAI(
        base_url=OPENROUTER_BASE_URL,
        api_key=resolved_key,
        default_headers={
            "X-Title":"rag-cag",
        },
    )
client=get_client()
print("client.base_url:", client.base_url)

client.base_url: https://openrouter.ai/api/v1/


## Step 2. `_build_messages()` — Chat Completions 메시지 페이로드 조립

### 목적 및 의의
* **중복 제거(DRY)**: `chat`과 `chat_structured`가 동일한 메시지 포맷 조립 로직을 공유하도록 내부 헬퍼로 분리한다.

### 레퍼런스 코드
```python
def _build_messages(prompt: str, system: str | None = None) -> list[dict]:
    messages: list[dict] = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    return messages


print(_build_messages("안녕", system="너는 연구 조수야."))
```

### 라인별 상세 해설
* `def _build_messages(...) -> list[dict]:`: 선행 언더스코어(`_`)는 모듈 외부 퍼블릭 API가 아닌 내부 전용 헬퍼임을 나타내는 파이썬 컨벤션.
* `if system: messages.append(...)`: `system` 프롬프트가 `None`이거나 빈 문자열일 때 불필요한 빈 system 메시지를 보내지 않도록 조건부 삽입.
* `messages.append({"role": "user", "content": prompt})`: 실제 질의 프롬프트를 마지막 `user` 턴으로 추가.

In [15]:
def _build_messages(prompt:str, system:str | None=None)->list[dict]:
    messages:list[dict]=[]
    if system:
        messages.append({'role':'system', 'content':system})
    messages.append({'role':'user', 'content':prompt})
    return messages

print(_build_messages('하이', ''))
print(_build_messages('하이', '넌 뛰어난 동료연구원이야.'))

[{'role': 'user', 'content': '하이'}]
[{'role': 'system', 'content': '넌 뛰어난 동료연구원이야.'}, {'role': 'user', 'content': '하이'}]


## Step 3. `chat()` — 기본 텍스트 Chat Completion 래퍼

### 목적 및 의의
* **의존성 주입(DI) & 기본값 캡슐화**: 매번 `client`를 넘기지 않아도 되지만, 커넥션 풀 재사용이나 테스트 목(Mock) 주입이 필요할 때는 `client` 파라미터로 외부 인스턴스를 받을 수 있게 설계한다.

### 레퍼런스 코드
```python
def chat(
    prompt: str,
    *,
    model: str = DEFAULT_MODEL,
    system: str | None = None,
    temperature: float = 0.2,
    max_tokens: int | None = None,
    client: OpenAI | None = None,
) -> str:
    active_client = client or get_client()
    response = active_client.chat.completions.create(
        model=model,
        messages=_build_messages(prompt, system),
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content or ""


print(chat("RAG와 CAG의 차이를 한 문장으로 설명해 줘.", max_tokens=150))
```

### 라인별 상세 해설
* `*,`: 이 기호 뒤의 모든 인자(`model`, `system`, `temperature`, `max_tokens`, `client`)를 **키워드 전용 인자(keyword-only arguments)**로 강제해 호출부 가독성을 높이고 인자 순서 실수를 원천 차단.
* `active_client = client or get_client()`: 외부에서 `client`를 넘겼으면 재사용(성능/테스트 유리), 안 넘겼으면 `get_client()`로 즉석 생성.
* `response.choices[0].message.content or ""`: 반환 타입 `-> str` 계약을 지키기 위해 `content`가 `None`인 예외 응답(예: tool call 전용 응답)에서도 빈 문자열로 안전하게 폴백.

In [20]:
import time
def chat(
    prompt:str,
    *,
    model:str=DEFAULT_MODEL,
    system:str | None=None,
    temperature:float=0.2,
    max_tokens:int | None=None,
    client:OpenAI | None=None,
) -> str:
    active_client=client or get_client()
    response=active_client.chat.completions.create(
        model=model,
        messages=_build_messages(prompt, system),
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content or ""
start=time.time()
print(chat("RAG 와 CAG 의 차이를 한 문장으로 설명해봐", max_tokens=150))
print(f'{time.time()-start:.5f}')

RAG(Retrieval-Augmented Generation)는 **외부 지식 검색을 통해 답변의 정확성과 풍부함을 높이는 방식**이고, CAG(Critique-Augmented Generation)는 **생성된 답변을 비판적으로 평가하고 수정하여 품질을 개선하는 방식**입니다.
2.66123


## Step 4. `chat_structured()` — Pydantic 기반 Structured Outputs 래퍼

### 목적 및 의의
* **타입 안전한 정보 추출**: Ontology 트리플 추출, 메타데이터 파싱 등에서 LLM 응답을 `BaseModel` 스키마로 강제·검증하여 후속 파이프라인의 파싱 에러를 제거한다.

### 레퍼런스 코드
```python
from pydantic import Field


def chat_structured(
    prompt: str,
    response_model: type[T],
    *,
    model: str = DEFAULT_MODEL,
    system: str | None = None,
    temperature: float = 0.0,
    client: OpenAI | None = None,
) -> T:
    active_client = client or get_client()
    response = active_client.chat.completions.parse(
        model=model,
        messages=_build_messages(prompt, system),
        response_format=response_model,
        temperature=temperature,
    )
    parsed = response.choices[0].message.parsed
    if parsed is None:
        raise ValueError("Model did not return a parsed structured response.")
    return parsed


class ConceptSummary(BaseModel):
    term: str = Field(description="핵심 용어")
    definition: str = Field(description="한 줄 정의")
    use_cases: list[str] = Field(description="대표 활용 사례 2~3개")


res = chat_structured("온톨로지(Ontology)를 요약해 줘.", ConceptSummary)
print(type(res), res.model_dump())
```

### 라인별 상세 해설
* `response_model: type[T]` -> `-> T`: 호출자가 `ConceptSummary` 클래스를 넘기면 반환 타입이 `BaseModel`이 아닌 정확한 `ConceptSummary`로 추론되도록 제네릭 `TypeVar T`를 사용.
* `active_client.chat.completions.parse(...)`: `response_format`에 전달된 Pydantic 클래스를 JSON Schema로 직렬화해 API에 전송하고, 응답 JSON을 자동으로 역직렬화·유효성 검증.
* `temperature: float = 0.0`: 구조화 추출에서는 무작위성보다 스키마 일관성과 재현성이 중요하므로 기본값을 `0.0`으로 고정.
* `if parsed is None: raise ValueError(...)`: 모델 거부(refusal)나 스키마 파싱 실패 시 `None`을 조용히 반환해 다운스트림에서 `AttributeError`가 터지는 대신, 원인 지점에서 즉시 명시적 예외 발생.

In [31]:
from pydantic import Field
def chat_structured(
    prompt: str,
    response_model:type[T],
    *,
    model:str=DEFAULT_MODEL,
    system:str|None=None,
    temperature:float=0.0,
    client:OpenAI|None=None,
)->T:
    active_client=client or get_client()
    response=active_client.chat.completions.parse(
        model=model,
        messages=_build_messages(prompt, system),
        response_format=response_model,
        temperature=temperature,
    )
    parsed=response.choices[0].message.parsed
    if parsed is None:
        raise ValueError("Model did not return a parsed structured response.")
    return parsed


class ConceptSummary(BaseModel):
    term: str = Field(description="핵심용어")
    definition: str=Field(description='한 줄 정의')
    use_cases:list[str] = Field(description='대표활용사례 2 ~ 3개')


res = chat_structured('온톨로지(Ontology)를 요약해 줘.', ConceptSummary)
print(type(res), res.model_dump())

<class '__main__.ConceptSummary'> {'term': '온톨로지(Ontology)', 'definition': '특정 도메인에서 사용되는 개념, 관계, 속성 등을 명확하게 정의하고 체계화한 지식 표현 모델입니다.', 'use_cases': ['시맨틱 웹 구축: 웹상의 정보를 기계가 이해하고 처리할 수 있도록 의미를 부여합니다.', '지식 관리 시스템: 기업이나 조직의 방대한 지식을 체계적으로 분류하고 검색 효율성을 높입니다.', '인공지능 및 전문가 시스템: 특정 분야의 전문 지식을 표현하여 추론 및 의사결정 지원에 활용됩니다.']}


In [30]:
c = 2
s = c or 100
s

2

## Step 5. `chat_vision()` — 멀티모달(OCR용) 로컬 이미지 질의 래퍼

### 목적 및 의의
* **로컬 파일 직접 전송**: 외부 이미지 호스팅(S3 등) 없이도 로컬 스캔 문서·스크린샷을 RFC 2397 Data URI(`data:<mime>;base64,...`)로 인코딩해 비전 LLM에 바로 전달한다.

### 레퍼런스 코드
```python
def chat_vision(
    prompt: str,
    image_path: str | Path,
    *,
    model: str = DEFAULT_MODEL,
    system: str | None = None,
    temperature: float = 0.0,
    client: OpenAI | None = None,
) -> str:
    path = Path(image_path)
    if not path.is_file():
        raise FileNotFoundError(f"Image not found: {path}")

    mime_type, _ = mimetypes.guess_type(path)
    resolved_mime = mime_type or "image/png"
    encoded = base64.b64encode(path.read_bytes()).decode("ascii")

    messages: list[dict] = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append(
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {
                    "type": "image_url",
                    "image_url": {"url": f"data:{resolved_mime};base64,{encoded}"},
                },
            ],
        }
    )

    active_client = client or get_client()
    response = active_client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
    )
    return response.choices[0].message.content or ""
```

### 라인별 상세 해설
* `path = Path(image_path)`: 호출자가 문자열 경로를 주든 `Path` 객체를 주든 일관된 `Path` 인터페이스로 정규화.
* `if not path.is_file(): raise FileNotFoundError(...)`: 존재하지 않는 파일이나 디렉토리 경로를 넘겼을 때 인코딩 단계 전에 조기 실패.
* `mime_type, _ = mimetypes.guess_type(path)`: 확장자 기반으로 `image/png`, `image/jpeg`, `image/webp` 등을 추론하고, 판별 불가 시 `image/png`로 보수적 폴백.
* `base64.b64encode(path.read_bytes()).decode("ascii")`: 바이너리 바이트열을 JSON 문자열에 안전하게 직렬화할 수 있는 Base64 ASCII 문자열로 변환.
* `"content": [{"type": "text", ...}, {"type": "image_url", ...}]`: OpenAI 멀티모달 메시지 규격은 `content`가 단일 문자열이 아니라 텍스트 파트와 이미지 파트의 배열(list) 구조.
* `f"data:{resolved_mime};base64,{encoded}"`: RFC 2397 Data URI 표준 포맷. 추후 `src/ocr/` 모듈에서 문서 이미지 OCR 실험을 할 때 이 함수를 그대로 호출하게 됨.

## Step 6. `src.core.llm` 실제 모듈 임포트 & 최종 대조 검증

### 목적 및 의의
* **노트북 구현과 `src/core/llm.py` 일치 확인**: 위에서 한 줄씩 구현한 모든 함수가 실제 프로젝트 모듈 `src/core/llm.py`에서 임포트했을 때도 동일하게 동작하는지 최종 검증한다.

### 레퍼런스 코드
```python
from src.core.llm import (
    DEFAULT_MODEL,
    OPENROUTER_BASE_URL,
    chat as pkg_chat,
    chat_structured as pkg_chat_structured,
    chat_vision as pkg_chat_vision,
    get_client as pkg_get_client,
)

print("Module constants:", OPENROUTER_BASE_URL, DEFAULT_MODEL)
print("Package chat():", pkg_chat("Say 'OK' only.", temperature=0.0, max_tokens=8))
print("Package chat_structured():", pkg_chat_structured("CAG를 요약해 줘.", ConceptSummary))
print("Package chat_vision callable:", callable(pkg_chat_vision))
```

### 라인별 상세 해설
* `as pkg_...` 별칭(alias) 사용 이유: Step 1~5에서 노트북 네임스페이스에 직접 정의한 `chat`, `get_client` 등과 이름 충돌 없이 실제 `src/core/llm.py`의 함수들을 분리 호출해 대조하기 위함.
* `callable(pkg_chat_vision)`: 테스트용 이미지가 아직 `data/`에 없더라도 함수 시그니처와 모듈 노출이 정상인지 확인 (이미지 파일을 `data/`에 넣으면 바로 `pkg_chat_vision("이 이미지의 텍스트를 추출해 줘", "data/sample.png")`로 테스트 가능).

## Step 7. OpenRouter 사용 가능 모델 리스트 조회 & 멀티 모델 비교 루프

### 목적 및 의의
* **최신 모델 ID 실시간 조회**: OpenRouter는 신규 모델이 수시로 추가·갱신되므로, 하드코딩된 목록에 의존하지 않고 공식 `/api/v1/models` 카탈로그 엔드포인트에서 현재 활성화된 모델 ID, 컨텍스트 길이(`context_length`), 입력/출력 단가(`pricing`)를 직접 조회한다.
* **태스크별 모델 벤치마크 기반**: 고른 모델 리스트를 동일한 프롬프트에 순차 호출하여 응답 품질과 지연시간(latency)을 정량 비교한다.

### 레퍼런스 코드
```python
import time
import httpx

# 1. OpenRouter 전체 모델 카탈로그 실시간 조회 (API 키 없이도 공개 조회 가능)
resp = httpx.get(f"{OPENROUTER_BASE_URL}/models", timeout=15.0)
resp.raise_for_status()
all_models = resp.json()["data"]
print(f"총 사용 가능 모델 수: {len(all_models)}")

# 2. 주요 프로바이더(google, anthropic, openai, deepseek, qwen) 중심으로 필터링하여 출력
target_prefixes = ("google/", "anthropic/", "openai/", "deepseek/", "qwen/")
filtered = [
    m for m in all_models
    if m["id"].startswith(target_prefixes)
]

# 컨텍스트 길이가 큰 순으로 상위 20개 출력 (RAG/CAG/OCR 후보 탐색용)
for m in sorted(filtered, key=lambda x: x.get("context_length", 0), reverse=True)[:20]:
    ctx = m.get("context_length", 0)
    prompt_price = float(m.get("pricing", {}).get("prompt", 0)) * 1_000_000
    print(f"{m['id']:<42} | ctx: {ctx:>8,} | $/1M in: ${prompt_price:.2f}")

# 3. 위 출력에서 마음에 드는 모델 ID를 골라 CANDIDATE_MODELS에 넣고 동일 프롬프트로 비교
CANDIDATE_MODELS = [
    "google/gemini-2.5-flash",
    "anthropic/claude-sonnet-4",
    "openai/gpt-4o-mini",
]

test_prompt = "CAG(Cache-Augmented Generation)의 핵심 전제 조건 한 가지를 한 문장으로 답해 줘."
for model_id in CANDIDATE_MODELS:
    t0 = time.perf_counter()
    try:
        out = pkg_chat(test_prompt, model=model_id, temperature=0.0, max_tokens=120)
        elapsed = time.perf_counter() - t0
        print(f"\n[{model_id}] ({elapsed:.2f}s)\n{out}")
    except Exception as exc:
        print(f"\n[{model_id}] ERROR: {exc}")
```

### 라인별 상세 해설
* `import time`: 각 모델 호출의 종단 간 응답 지연시간(`perf_counter`)을 정밀 측정하기 위해 임포트.
* `import httpx`: `pyproject.toml`에 이미 포함된 비동기/동기 HTTP 클라이언트로, OpenAI SDK 외부의 OpenRouter 메타데이터 엔드포인트(`/models`)를 직접 호출할 때 사용.
* `httpx.get(f"{OPENROUTER_BASE_URL}/models", timeout=15.0)`: OpenRouter가 제공하는 전체 모델 카탈로그 JSON(`{"data": [...]}`)을 가져오며, 네트워크 지연 시 무한 대기를 막기 위해 15초 타임아웃 지정.
* `resp.raise_for_status()`: 4xx/5xx HTTP 오류 시 조용히 빈 dict를 파싱하지 않고 즉시 `HTTPStatusError`를 발생시키는 방어 코드.
* `m["id"].startswith(target_prefixes)`: 수백 개의 커뮤니티·파인튜닝 모델 중 주요 상용/오픈웨이트 계열만 추려 가독성 확보.
* `float(m.get("pricing", {}).get("prompt", 0)) * 1_000_000`: OpenRouter는 토큰 1개당 달러 문자열로 단가를 반환하므로, 실무 비교 단위인 **1M(백만) 토큰당 달러($/1M)**로 환산.
* `CANDIDATE_MODELS`: 위 조회 결과에서 선택한 모델 ID들을 리스트로 관리하고, `pkg_chat(..., model=model_id)`의 `model` 인자만 교체해 동일 조건(`temperature=0.0`)으로 비교.
* `try ... except Exception`: 특정 모델 ID가 계정의 크레딧/권한 제약이나 일시적 장애로 실패하더라도 루프 전체가 중단되지 않고 나머지 후보 모델 결과를 끝까지 수집.